<a href="https://colab.research.google.com/github/tiagoeletro-bot/Ar-Condicionado-HVAC---notebookLM-/blob/main/Projeto_Alarme.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ============================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
# MÓDULO 1 - IMPORTAÇÃO, CONSOLIDAÇÃO E QUALIDADE DOS DADOS
# ============================================================

# ============================================================
# 1. BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ============================================================
# 2. UPLOAD DO ARQUIVO
# ============================================================

from google.colab import files

print("=" * 80)
print("UPLOAD DA BASE DE ALARMES METASYS")
print("=" * 80)

uploaded = files.upload()

ARQUIVO = "02 - Alarmes Metasys 2026 R0.xlsx"

if not Path(ARQUIVO).exists():
    raise FileNotFoundError(
        f"\nO arquivo esperado '{ARQUIVO}' não foi localizado.\n"
        "Verifique se o nome do arquivo enviado ao Colab é exatamente esse."
    )

print(f"\nArquivo localizado com sucesso: {ARQUIVO}")


# ============================================================
# 3. DICIONÁRIOS DE REFERÊNCIA
# ============================================================

# ATENÇÃO:
# Estes dicionários serão utilizados apenas para AUDITORIA.
# Eles NÃO irão sobrescrever as classificações existentes
# nas colunas Categoria e Evento.

MAPA_CATEGORIA_INFORMADO = {
    0: "HVAC",
    1: "SDAI",
    2: "Segurança",
    3: "Serviço",
    4: "Administrativo",
    5: "Geral",
    6: "Iluminação",
    7: "Refrigeração",
    8: "Ambiente Crítico",
    9: "Qualidade do Ar",
    10: "Potência",
    11: "Energia",
    12: "Sistema",
    100: "Controle de Demanda",
    101: "Hidráulica"
}


MAPA_STATUS_INFORMADO = {
    2: "Normal",
    28: "Alarme Alto",
    18: "Alarme Baixo",
    14: "Pré Alarme Alto",
    3: "Pré Alarme Baixo",
    120: "Unreliable",
    106: "Offline",
    4: "Sistema",
    5: "Sistema",
    554: "Sistema",
    872: "SDAI",
    3267: "Sistema",
    22: "Alarme"
}


MAPA_MESES = {
    1: "Janeiro",
    2: "Fevereiro",
    3: "Março",
    4: "Abril",
    5: "Maio",
    6: "Junho",
    7: "Julho",
    8: "Agosto",
    9: "Setembro",
    10: "Outubro",
    11: "Novembro",
    12: "Dezembro"
}


MAPA_DIA_SEMANA = {
    1: "Segunda",
    2: "Terça",
    3: "Quarta",
    4: "Quinta",
    5: "Sexta",
    6: "Sábado",
    7: "Domingo"
}


# ============================================================
# 4. INSPEÇÃO DO ARQUIVO
# ============================================================

print("\n" + "=" * 80)
print("INSPEÇÃO DAS ABAS")
print("=" * 80)

excel = pd.ExcelFile(ARQUIVO)

abas = excel.sheet_names

print(f"\nQuantidade de abas encontradas: {len(abas)}")

for aba in abas:
    print(f" - {aba}")


# ============================================================
# 5. COLUNAS ESPERADAS
# ============================================================

COLUNAS_ESPERADAS = [
    "utcCreationDateTime",
    "priority",
    "currentStatusEnumInfoId",
    "itemName",
    "itemDescription",
    "itemCategoryEnumInfoId",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",
    "previousStatusEnumInfoId"
]


# ============================================================
# 6. LEITURA E CONSOLIDAÇÃO DAS ABAS
# ============================================================

print("\n" + "=" * 80)
print("IMPORTAÇÃO E CONSOLIDAÇÃO")
print("=" * 80)

lista_dfs = []
resumo_importacao = []

for aba in abas:

    print(f"Lendo aba: {aba}")

    df_temp = pd.read_excel(
        ARQUIVO,
        sheet_name=aba
    )

    # Padronização dos nomes das colunas
    df_temp.columns = df_temp.columns.str.strip()

    colunas_faltantes = [
        col
        for col in COLUNAS_ESPERADAS
        if col not in df_temp.columns
    ]

    colunas_extras = [
        col
        for col in df_temp.columns
        if col not in COLUNAS_ESPERADAS
    ]

    resumo_importacao.append({
        "Aba": aba,
        "Registros": len(df_temp),
        "Colunas": len(df_temp.columns),
        "Colunas_Faltantes": ", ".join(colunas_faltantes),
        "Colunas_Extras": ", ".join(colunas_extras)
    })

    # Identificação da aba de origem
    df_temp["Aba_Origem"] = aba

    lista_dfs.append(df_temp)


# Consolidação
df = pd.concat(
    lista_dfs,
    ignore_index=True
)

resumo_importacao = pd.DataFrame(resumo_importacao)

print("\nImportação concluída.")
print(f"Total de registros consolidados: {len(df):,}")
print(f"Total de colunas: {df.shape[1]}")


# ============================================================
# 7. RESUMO DA IMPORTAÇÃO
# ============================================================

print("\n" + "=" * 80)
print("RESUMO DA IMPORTAÇÃO")
print("=" * 80)

display(resumo_importacao)


# ============================================================
# 8. PADRONIZAÇÃO DOS TIPOS DE DADOS
# ============================================================

print("\nPadronizando tipos de dados...")


# Data/Hora
df["utcCreationDateTime"] = pd.to_datetime(
    df["utcCreationDateTime"],
    errors="coerce"
)


# Colunas numéricas
colunas_numericas = [
    "priority",
    "currentStatusEnumInfoId",
    "itemCategoryEnumInfoId",
    "previousStatusEnumInfoId"
]

for coluna in colunas_numericas:

    df[coluna] = pd.to_numeric(
        df[coluna],
        errors="coerce"
    )


# Colunas textuais
colunas_texto = [
    "itemName",
    "itemDescription",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]

for coluna in colunas_texto:

    df[coluna] = (
        df[coluna]
        .astype("string")
        .str.strip()
        .replace(
            ["", "nan", "None", "<NA>"],
            pd.NA
        )
    )

print("Tipos de dados padronizados.")


# ============================================================
# 9. CRIAÇÃO DE DATA/HORA UTC E LOCAL
# ============================================================

print("\nConvertendo horário UTC para America/Sao_Paulo...")


df["DataHoraUTC"] = df["utcCreationDateTime"]


# Converte somente se a coluna estiver sem timezone
if df["DataHoraUTC"].dt.tz is None:

    df["DataHoraLocal"] = (
        df["DataHoraUTC"]
        .dt.tz_localize(
            "UTC",
            ambiguous="NaT",
            nonexistent="NaT"
        )
        .dt.tz_convert(
            "America/Sao_Paulo"
        )
    )

else:

    df["DataHoraLocal"] = (
        df["DataHoraUTC"]
        .dt.tz_convert(
            "America/Sao_Paulo"
        )
    )


print("Conversão concluída.")


# ============================================================
# 10. CRIAÇÃO DAS VARIÁVEIS TEMPORAIS
# ============================================================

df["Data"] = (
    df["DataHoraLocal"]
    .dt.date
)

df["Ano"] = (
    df["DataHoraLocal"]
    .dt.year
)

df["Mes_Numero"] = (
    df["DataHoraLocal"]
    .dt.month
)

df["Mes"] = (
    df["Mes_Numero"]
    .map(MAPA_MESES)
)

df["Dia"] = (
    df["DataHoraLocal"]
    .dt.day
)

df["Hora"] = (
    df["DataHoraLocal"]
    .dt.hour
)

df["Minuto"] = (
    df["DataHoraLocal"]
    .dt.minute
)

df["Dia_Semana_Numero"] = (
    df["DataHoraLocal"]
    .dt.dayofweek + 1
)

df["Dia_Semana"] = (
    df["Dia_Semana_Numero"]
    .map(MAPA_DIA_SEMANA)
)


# ============================================================
# 11. CLASSIFICAÇÃO DO PERÍODO DO DIA
# ============================================================

def classificar_periodo(hora):

    if pd.isna(hora):
        return pd.NA

    if 0 <= hora < 6:
        return "Madrugada"

    elif 6 <= hora < 12:
        return "Manhã"

    elif 12 <= hora < 18:
        return "Tarde"

    else:
        return "Noite"


df["Periodo_Dia"] = (
    df["Hora"]
    .apply(classificar_periodo)
)


# ============================================================
# 12. FLAGS DE QUALIDADE
# ============================================================

df["Flag_ItemName_Vazio"] = (
    df["itemName"]
    .isna()
)

df["Flag_Descricao_Vazia"] = (
    df["itemDescription"]
    .isna()
)

df["Flag_Categoria_Vazia"] = (
    df["Categoria"]
    .isna()
)

df["Flag_Evento_Vazio"] = (
    df["Evento"]
    .isna()
)

df["Flag_Mantenedor_Vazio"] = (
    df["Mantenedor"]
    .isna()
)

df["Flag_Tipo_Vazio"] = (
    df["Tipo"]
    .isna()
)

df["Flag_Torre_Vazia"] = (
    df["Torre"]
    .isna()
)


# ============================================================
# 13. IDENTIFICAÇÃO DE POSSÍVEIS DUPLICIDADES
# ============================================================

COLUNAS_CHAVE_DUPLICIDADE = [
    "utcCreationDateTime",
    "priority",
    "currentStatusEnumInfoId",
    "itemName",
    "previousStatusEnumInfoId"
]


df["Flag_Duplicado"] = (
    df
    .duplicated(
        subset=COLUNAS_CHAVE_DUPLICIDADE,
        keep=False
    )
)


qtd_duplicados = (
    df["Flag_Duplicado"]
    .sum()
)

perc_duplicados = (
    qtd_duplicados
    / len(df)
    * 100
)

print("\n" + "=" * 80)
print("DUPLICIDADES")
print("=" * 80)

print(f"Possíveis registros duplicados: {qtd_duplicados:,}")
print(f"Percentual de possíveis duplicados: {perc_duplicados:.4f}%")


# ============================================================
# 14. AUDITORIA itemCategoryEnumInfoId × Categoria
# ============================================================

auditoria_categoria = (
    df
    .groupby(
        [
            "itemCategoryEnumInfoId",
            "Categoria"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


auditoria_categoria["Categoria_Dicionario_Informado"] = (
    auditoria_categoria[
        "itemCategoryEnumInfoId"
    ]
    .map(MAPA_CATEGORIA_INFORMADO)
)


# Começa classificando tudo como divergente
auditoria_categoria["Validacao_Dicionario"] = "Divergente"

# Código sem correspondência no dicionário informado
mascara_sem_codigo = (
    auditoria_categoria[
        "Categoria_Dicionario_Informado"
    ].isna()
)

auditoria_categoria.loc[
    mascara_sem_codigo,
    "Validacao_Dicionario"
] = "Código não existente no dicionário informado"


# Código e descrição coincidentes
mascara_coincidente = (
    auditoria_categoria["Categoria"].fillna("")
    ==
    auditoria_categoria[
        "Categoria_Dicionario_Informado"
    ].fillna("")
)

auditoria_categoria.loc[
    mascara_coincidente,
    "Validacao_Dicionario"
] = "Coincidente"


print("\n" + "=" * 80)
print("AUDITORIA DE CATEGORIAS")
print("=" * 80)

display(
    auditoria_categoria.head(100)
)


# ============================================================
# 15. AUDITORIA currentStatusEnumInfoId × Evento
# ============================================================

auditoria_status = (
    df
    .groupby(
        [
            "currentStatusEnumInfoId",
            "Evento"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


auditoria_status["Evento_Dicionario_Informado"] = (
    auditoria_status[
        "currentStatusEnumInfoId"
    ]
    .map(MAPA_STATUS_INFORMADO)
)


# Começa classificando tudo como divergente
auditoria_status["Validacao_Dicionario"] = "Divergente"

# Código sem correspondência no dicionário informado
mascara_sem_codigo = (
    auditoria_status[
        "Evento_Dicionario_Informado"
    ].isna()
)

auditoria_status.loc[
    mascara_sem_codigo,
    "Validacao_Dicionario"
] = "Código não existente no dicionário informado"


# Código e descrição coincidentes
mascara_coincidente = (
    auditoria_status["Evento"].fillna("")
    ==
    auditoria_status[
        "Evento_Dicionario_Informado"
    ].fillna("")
)

auditoria_status.loc[
    mascara_coincidente,
    "Validacao_Dicionario"
] = "Coincidente"


print("\n" + "=" * 80)
print("AUDITORIA DE STATUS / EVENTOS")
print("=" * 80)

display(
    auditoria_status.head(100)
)


# ============================================================
# 16. CONSISTÊNCIA DOS CÓDIGOS DE CATEGORIA
# ============================================================

consistencia_categoria = (
    df
    .groupby(
        "itemCategoryEnumInfoId",
        observed=True
    )["Categoria"]
    .nunique()
    .reset_index(
        name="Qtd_Categorias_Diferentes"
    )
    .sort_values(
        "Qtd_Categorias_Diferentes",
        ascending=False
    )
)


# ============================================================
# 17. CONSISTÊNCIA DOS CÓDIGOS DE EVENTO
# ============================================================

consistencia_evento = (
    df
    .groupby(
        "currentStatusEnumInfoId",
        observed=True
    )["Evento"]
    .nunique()
    .reset_index(
        name="Qtd_Eventos_Diferentes"
    )
    .sort_values(
        "Qtd_Eventos_Diferentes",
        ascending=False
    )
)


# ============================================================
# 18. DIAGNÓSTICO DAS DIMENSÕES
# ============================================================

DIMENSOES = [
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]


diagnosticos_dimensoes = {}


print("\n" + "=" * 80)
print("DIAGNÓSTICO DAS DIMENSÕES")
print("=" * 80)


for coluna in DIMENSOES:

    resultado = (
        df[coluna]
        .value_counts(
            dropna=False
        )
        .reset_index()
    )

    resultado.columns = [
        coluna,
        "Quantidade"
    ]

    resultado["Percentual"] = (
        resultado["Quantidade"]
        / len(df)
        * 100
    )

    diagnosticos_dimensoes[coluna] = resultado

    print("\n" + "-" * 80)
    print(f"COLUNA: {coluna}")
    print("-" * 80)

    display(
        resultado.head(50)
    )


# ============================================================
# 19. RELATÓRIO DE QUALIDADE GERAL
# ============================================================

qualidade = []


for coluna in df.columns:

    quantidade_nulos = (
        df[coluna]
        .isna()
        .sum()
    )

    percentual_nulos = (
        quantidade_nulos
        / len(df)
        * 100
    )

    quantidade_unicos = (
        df[coluna]
        .nunique(
            dropna=True
        )
    )

    qualidade.append({
        "Coluna": coluna,
        "Tipo_Dado": str(df[coluna].dtype),
        "Registros": len(df),
        "Nulos": quantidade_nulos,
        "Percentual_Nulos": percentual_nulos,
        "Valores_Unicos": quantidade_unicos
    })


df_qualidade = (
    pd.DataFrame(
        qualidade
    )
    .sort_values(
        "Percentual_Nulos",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("QUALIDADE GERAL DA BASE")
print("=" * 80)

display(df_qualidade)


# ============================================================
# 20. AUDITORIA DAS PRIORIDADES
# ============================================================

prioridades = (
    df["priority"]
    .value_counts(
        dropna=False
    )
    .sort_index()
    .reset_index()
)

prioridades.columns = [
    "Priority",
    "Quantidade"
]

prioridades["Percentual"] = (
    prioridades["Quantidade"]
    / len(df)
    * 100
)


print("\n" + "=" * 80)
print("DISTRIBUIÇÃO DAS PRIORIDADES")
print("=" * 80)

display(prioridades)


# ============================================================
# 21. RELAÇÃO TORRE × TTL TORRE
# ============================================================

torre_ttl = (
    df
    .groupby(
        [
            "Torre",
            "TTL Torre"
        ],
        dropna=False,
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
    .sort_values(
        "Quantidade",
        ascending=False
    )
)


consistencia_torre = (
    df
    .groupby(
        "Torre",
        observed=True
    )["TTL Torre"]
    .nunique()
    .reset_index(
        name="Qtd_TTL_Torre"
    )
    .sort_values(
        "Qtd_TTL_Torre",
        ascending=False
    )
)


print("\n" + "=" * 80)
print("RELAÇÃO TORRE × TTL TORRE")
print("=" * 80)

display(
    torre_ttl.head(100)
)


# ============================================================
# 22. RESUMO EXECUTIVO DA BASE
# ============================================================

print("\n" + "=" * 80)
print("RESUMO EXECUTIVO DA BASE")
print("=" * 80)

print(f"Registros totais.............: {len(df):,}")

print(
    f"Período inicial..............: "
    f"{df['DataHoraLocal'].min()}"
)

print(
    f"Período final................: "
    f"{df['DataHoraLocal'].max()}"
)

print(
    f"Equipamentos únicos..........: "
    f"{df['itemName'].nunique():,}"
)

print(
    f"Descrições únicas............: "
    f"{df['itemDescription'].nunique():,}"
)

print(
    f"Categorias...................: "
    f"{df['Categoria'].nunique():,}"
)

print(
    f"Eventos......................: "
    f"{df['Evento'].nunique():,}"
)

print(
    f"Mantenedores.................: "
    f"{df['Mantenedor'].nunique():,}"
)

print(
    f"Tipos........................: "
    f"{df['Tipo'].nunique():,}"
)

print(
    f"Torres.......................: "
    f"{df['Torre'].nunique():,}"
)

print(
    f"Prioridades diferentes.......: "
    f"{df['priority'].nunique():,}"
)

print(
    f"Possíveis duplicidades.......: "
    f"{qtd_duplicados:,}"
)

print("=" * 80)


# ============================================================
# 23. ORDENAÇÃO CRONOLÓGICA
# ============================================================

print("\nOrdenando a base cronologicamente...")


df = (
    df
    .sort_values(
        [
            "itemName",
            "DataHoraUTC"
        ]
    )
    .reset_index(
        drop=True
    )
)


print("Ordenação concluída.")


# ============================================================
# 24. OTIMIZAÇÃO DE MEMÓRIA
# ============================================================

memoria_antes = (
    df
    .memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


colunas_categoria = [
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",
    "Aba_Origem",
    "Dia_Semana",
    "Periodo_Dia",
    "Mes"
]


for coluna in colunas_categoria:

    if coluna in df.columns:

        df[coluna] = (
            df[coluna]
            .astype("category")
        )


memoria_depois = (
    df
    .memory_usage(
        deep=True
    )
    .sum()
    / 1024**2
)


print("\n" + "=" * 80)
print("OTIMIZAÇÃO DE MEMÓRIA")
print("=" * 80)

print(
    f"Memória antes da otimização.: "
    f"{memoria_antes:,.2f} MB"
)

print(
    f"Memória após otimização......: "
    f"{memoria_depois:,.2f} MB"
)

print(
    f"Redução......................: "
    f"{((memoria_antes - memoria_depois) / memoria_antes) * 100:.2f}%"
)


# ============================================================
# 25. SALVAMENTO DA BASE TRATADA EM PARQUET
# ============================================================

ARQUIVO_PARQUET = (
    "Base_Alarmes_Metasys_2026_Tratada.parquet"
)


print("\nSalvando base tratada em formato Parquet...")


df.to_parquet(
    ARQUIVO_PARQUET,
    index=False
)


print(
    f"Base tratada salva com sucesso:\n"
    f"{ARQUIVO_PARQUET}"
)


# ============================================================
# 26. EXPORTAÇÃO DO RELATÓRIO DE QUALIDADE
# ============================================================

ARQUIVO_QUALIDADE = (
    "Relatorio_Qualidade_Alarmes_Metasys_2026.xlsx"
)


print("\nGerando relatório de qualidade em Excel...")


with pd.ExcelWriter(
    ARQUIVO_QUALIDADE,
    engine="openpyxl"
) as writer:

    resumo_importacao.to_excel(
        writer,
        sheet_name="Importacao",
        index=False
    )

    df_qualidade.to_excel(
        writer,
        sheet_name="Qualidade",
        index=False
    )

    auditoria_categoria.to_excel(
        writer,
        sheet_name="Auditoria Categoria",
        index=False
    )

    auditoria_status.to_excel(
        writer,
        sheet_name="Auditoria Status",
        index=False
    )

    consistencia_categoria.to_excel(
        writer,
        sheet_name="Consist Categoria",
        index=False
    )

    consistencia_evento.to_excel(
        writer,
        sheet_name="Consist Evento",
        index=False
    )

    prioridades.to_excel(
        writer,
        sheet_name="Prioridades",
        index=False
    )

    torre_ttl.to_excel(
        writer,
        sheet_name="Torre x TTL",
        index=False
    )

    consistencia_torre.to_excel(
        writer,
        sheet_name="Consist Torre",
        index=False
    )

    for nome, tabela in diagnosticos_dimensoes.items():

        nome_aba = (
            f"Diag {nome}"
            [:31]
        )

        tabela.to_excel(
            writer,
            sheet_name=nome_aba,
            index=False
        )


print(
    f"Relatório de qualidade criado:\n"
    f"{ARQUIVO_QUALIDADE}"
)


# ============================================================
# 27. AMOSTRA FINAL DA BASE TRATADA
# ============================================================

print("\n" + "=" * 80)
print("AMOSTRA DA BASE FINAL TRATADA")
print("=" * 80)

colunas_amostra = [
    "DataHoraUTC",
    "DataHoraLocal",
    "Mes",
    "Dia_Semana",
    "Hora",
    "Periodo_Dia",
    "priority",
    "currentStatusEnumInfoId",
    "previousStatusEnumInfoId",
    "itemName",
    "itemDescription",
    "itemCategoryEnumInfoId",
    "Categoria",
    "Evento",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre"
]


display(
    df[
        colunas_amostra
    ].head(20)
)


# ============================================================
# 28. DOWNLOAD DOS ARQUIVOS GERADOS
# ============================================================

print("\n" + "=" * 80)
print("PROCESSAMENTO CONCLUÍDO COM SUCESSO")
print("=" * 80)

print("\nArquivos gerados:")

print(
    f"1. {ARQUIVO_PARQUET}"
)

print(
    f"2. {ARQUIVO_QUALIDADE}"
)


print(
    "\nIniciando download do relatório de qualidade..."
)

files.download(
    ARQUIVO_QUALIDADE
)


print(
    "\nOBSERVAÇÃO:"
    "\nA base Parquet pode ser baixada executando:"
    "\nfiles.download(ARQUIVO_PARQUET)"
)


UPLOAD DA BASE DE ALARMES METASYS


Saving 02 - Alarmes Metasys 2026 R0.xlsx to 02 - Alarmes Metasys 2026 R0 (1).xlsx

Arquivo localizado com sucesso: 02 - Alarmes Metasys 2026 R0.xlsx

INSPEÇÃO DAS ABAS

Quantidade de abas encontradas: 7
 - Janeiro
 - Fevereiro
 - Março
 - Abril
 - Maio
 - Junho
 - Julho

IMPORTAÇÃO E CONSOLIDAÇÃO
Lendo aba: Janeiro
Lendo aba: Fevereiro
Lendo aba: Março
Lendo aba: Abril
Lendo aba: Maio
Lendo aba: Junho
Lendo aba: Julho

Importação concluída.
Total de registros consolidados: 1,413,882
Total de colunas: 14

RESUMO DA IMPORTAÇÃO


,Aba,Registros,Colunas,Colunas_Faltantes,Colunas_Extras
0,Janeiro,294183,13,,
1,Fevereiro,218226,13,,
2,Março,207241,13,,
3,Abril,180463,13,,
4,Maio,157589,13,,
5,Junho,174446,13,,
6,Julho,181734,13,,



Padronizando tipos de dados...
Tipos de dados padronizados.

Convertendo horário UTC para America/Sao_Paulo...
Conversão concluída.

DUPLICIDADES
Possíveis registros duplicados: 12,118
Percentual de possíveis duplicados: 0.8571%

AUDITORIA DE CATEGORIAS


,itemCategoryEnumInfoId,Categoria,Quantidade,Categoria_Dicionario_Informado,Validacao_Dicionario
2,16,HVAC,1067619,NaN,Código não existente no dicionário informado
1,12,Sistema,193061,Sistema,Coincidente
0,8,Geral,89192,Ambiente Crítico,Divergente
6,285,Hidráulica,37499,NaN,Código não existente no dicionário informado
7,387,Iluminação,24617,NaN,Código não existente no dicionário informado
5,153,Energia,1773,NaN,Código não existente no dicionário informado
4,129,SDAI,66,NaN,Código não existente no dicionário informado
3,84,Potência,40,NaN,Código não existente no dicionário informado
8,2184,Administrativo,15,NaN,Código não existente no dicionário informado


...

,DataHoraUTC,DataHoraLocal,Mes,Dia_Semana,Hora,Periodo_Dia,priority,currentStatusEnumInfoId,previousStatusEnumInfoId,itemName,itemDescription,itemCategoryEnumInfoId,Categoria,Evento,Mantenedor,Tipo,Torre,TTL Torre
0,2026-05-28 21:52:33,2026-05-28 18:52:33-03:00,Maio,Quinta,18,Noite,106,22,2,89-CGM09090 Default State,<NA>,8,Geral,Off-line,Others,Off-line,TAE,CEITOB
1,2026-01-01 09:41:46,2026-01-01 06:41:46-03:00,Janeiro,Quinta,6,Manhã,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
2,2026-01-01 10:31:48,2026-01-01 07:31:48-03:00,Janeiro,Quinta,7,Manhã,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
3,2026-01-03 08:38:33,2026-01-03 05:38:33-03:00,Janeiro,Sábado,5,Madrugada,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
4,2026-01-03 09:28:33,2026-01-03 06:28:33-03:00,Janeiro,Sábado,6,Manhã,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
5,2026-01-16 17:59:12,2026-01-16 14:59:12-03:00,Janeiro,Sexta,14,Tarde,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
6,2026-01-16 18:49:08,2026-01-16 15:49:08-03:00,Janeiro,Sexta,15,Tarde,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
7,2026-01-18 01:40:44,2026-01-17 22:40:44-03:00,Janeiro,Sábado,22,Noite,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6
8,2026-01-18 02:30:40,2026-01-17 23:30:40-03:00,Janeiro,Sábado,23,Noite,200,2,22,Alarme_BB_TAE,<NA>,285,Hidráulica,Normal,Hidráulica,Other,Bloco E6,CEITE6
9,2026-01-18 03:58:33,2026-01-18 00:58:33-03:00,Janeiro,Domingo,0,Madrugada,70,22,2,Alarme_BB_TAE,<NA>,285,Hidráulica,Alarme,Hidráulica,Other,Bloco E6,CEITE6



PROCESSAMENTO CONCLUÍDO COM SUCESSO

Arquivos gerados:
1. Base_Alarmes_Metasys_2026_Tratada.parquet
2. Relatorio_Qualidade_Alarmes_Metasys_2026.xlsx

Iniciando download do relatório de qualidade...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


OBSERVAÇÃO:
A base Parquet pode ser baixada executando:
files.download(ARQUIVO_PARQUET)


In [6]:
# ============================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
# MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES DE ESTADO
# ============================================================

# ============================================================
# 1. BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
import unicodedata
import re
from pathlib import Path
from IPython.display import display
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 150)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# ============================================================
# 2. ARQUIVOS
# ============================================================

ARQUIVO_ENTRADA = "Base_Alarmes_Metasys_2026_Tratada.parquet"

ARQUIVO_SAIDA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo2_Classificacao_Alarmes_Metasys.xlsx"
)


# ============================================================
# 3. VERIFICAÇÃO DO ARQUIVO
# ============================================================

print("=" * 90)
print("MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES")
print("=" * 90)

if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' não foi localizado.\n"
        "Faça o upload do arquivo Parquet gerado no Módulo 1."
    )


# ============================================================
# 4. LEITURA DA BASE
# ============================================================

print("\nCarregando base tratada do Módulo 1...")

df = pd.read_parquet(
    ARQUIVO_ENTRADA
)

print("Base carregada com sucesso.")
print(f"Registros: {len(df):,}")
print(f"Colunas iniciais: {df.shape[1]}")


# ============================================================
# 5. VERIFICAÇÃO DAS COLUNAS NECESSÁRIAS
# ============================================================

COLUNAS_NECESSARIAS = [
    "DataHoraUTC",
    "DataHoraLocal",
    "priority",
    "currentStatusEnumInfoId",
    "previousStatusEnumInfoId",
    "itemName",
    "Evento",
    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre"
]

faltantes = [
    coluna
    for coluna in COLUNAS_NECESSARIAS
    if coluna not in df.columns
]

if faltantes:

    raise ValueError(
        "As seguintes colunas necessárias não foram encontradas:\n"
        + "\n".join(faltantes)
    )


# ============================================================
# 6. FUNÇÃO DE NORMALIZAÇÃO DE TEXTO
# ============================================================

def normalizar_texto(valor):

    if pd.isna(valor):
        return ""

    texto = str(valor).strip().lower()

    texto = unicodedata.normalize(
        "NFKD",
        texto
    )

    texto = "".join(
        caractere
        for caractere in texto
        if not unicodedata.combining(caractere)
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto
    )

    return texto


# ============================================================
# 7. DICIONÁRIO EMPÍRICO:
#    currentStatusEnumInfoId → Evento mais frequente
# ============================================================

print("\nConstruindo dicionário empírico de códigos de estado...")


mapa_status_empirico_df = (
    df[
        [
            "currentStatusEnumInfoId",
            "Evento"
        ]
    ]
    .dropna(
        subset=[
            "currentStatusEnumInfoId",
            "Evento"
        ]
    )
    .groupby(
        [
            "currentStatusEnumInfoId",
            "Evento"
        ],
        observed=True
    )
    .size()
    .reset_index(
        name="Quantidade"
    )
)


# Evento predominante para cada código
mapa_status_principal = (
    mapa_status_empirico_df
    .sort_values(
        [
            "currentStatusEnumInfoId",
            "Quantidade"
        ],
        ascending=[
            True,
            False
        ]
    )
    .drop_duplicates(
        subset="currentStatusEnumInfoId",
        keep="first"
    )
)


MAPA_STATUS_EMPIRICO = dict(
    zip(
        mapa_status_principal[
            "currentStatusEnumInfoId"
        ],
        mapa_status_principal[
            "Evento"
        ].astype(str)
    )
)


print(
    f"Códigos mapeados empiricamente: "
    f"{len(MAPA_STATUS_EMPIRICO):,}"
)


# ============================================================
# 8. EVENTO ATUAL E EVENTO ANTERIOR
# ============================================================

# Evento atual da própria base
df["Evento_Atual"] = (
    df["Evento"]
    .astype("string")
)


# Evento anterior reconstruído através do código anterior
df["Evento_Anterior"] = (
    df[
        "previousStatusEnumInfoId"
    ]
    .map(
        MAPA_STATUS_EMPIRICO
    )
    .astype("string")
)


# Flag para códigos anteriores sem tradução possível
df["Flag_Evento_Anterior_Desconhecido"] = (
    df["previousStatusEnumInfoId"].notna()
    &
    df["Evento_Anterior"].isna()
)


# ============================================================
# 9. CLASSIFICAÇÃO DA FAMÍLIA DO EVENTO
# ============================================================

def classificar_familia_evento(evento):

    texto = normalizar_texto(evento)

    if texto == "":
        return "Desconhecido"

    # --------------------------------------------------------
    # NORMAL
    # --------------------------------------------------------

    if texto == "normal":
        return "Normal"

    # --------------------------------------------------------
    # OFFLINE
    # --------------------------------------------------------

    if (
        "offline" in texto
        or "off-line" in texto
    ):
        return "Offline"

    # --------------------------------------------------------
    # UNRELIABLE
    # --------------------------------------------------------

    if (
        "unreliable" in texto
        or "nao confiavel" in texto
    ):
        return "Unreliable"

    # --------------------------------------------------------
    # PRÉ-ALARME
    # Tem que vir antes de "alarme"
    # --------------------------------------------------------

    if (
        "pre alarme" in texto
        or "pre-alarme" in texto
    ):

        if (
            "alta" in texto
            or "alto" in texto
        ):
            return "Pré-Alarme Alto"

        if (
            "baixa" in texto
            or "baixo" in texto
        ):
            return "Pré-Alarme Baixo"

        return "Pré-Alarme"

    # --------------------------------------------------------
    # ALARME
    # --------------------------------------------------------

    if "alarme" in texto:

        if (
            "alta" in texto
            or "alto" in texto
        ):
            return "Alarme Alto"

        if (
            "baixa" in texto
            or "baixo" in texto
        ):
            return "Alarme Baixo"

        return "Alarme"

    # --------------------------------------------------------
    # SDAI
    # --------------------------------------------------------

    if "sdai" in texto:
        return "SDAI"

    # --------------------------------------------------------
    # SISTEMA
    # --------------------------------------------------------

    if "sistema" in texto:
        return "Sistema"

    # --------------------------------------------------------
    # DEMAIS SITUAÇÕES
    # --------------------------------------------------------

    return "Outros"


df["Familia_Evento_Atual"] = (
    df["Evento_Atual"]
    .apply(
        classificar_familia_evento
    )
)


df["Familia_Evento_Anterior"] = (
    df["Evento_Anterior"]
    .apply(
        classificar_familia_evento
    )
)


# ============================================================
# 10. GRUPOS MACRO DOS EVENTOS
# ============================================================

def classificar_grupo_macro(familia):

    if familia == "Normal":
        return "Normal"

    if familia in [
        "Alarme",
        "Alarme Alto",
        "Alarme Baixo"
    ]:
        return "Alarme"

    if familia in [
        "Pré-Alarme",
        "Pré-Alarme Alto",
        "Pré-Alarme Baixo"
    ]:
        return "Pré-Alarme"

    if familia in [
        "Offline",
        "Unreliable"
    ]:
        return "Comunicação / Confiabilidade"

    if familia == "Sistema":
        return "Sistema"

    if familia == "SDAI":
        return "SDAI"

    if familia == "Desconhecido":
        return "Desconhecido"

    return "Outros"


df["Grupo_Evento_Atual"] = (
    df["Familia_Evento_Atual"]
    .apply(
        classificar_grupo_macro
    )
)


df["Grupo_Evento_Anterior"] = (
    df["Familia_Evento_Anterior"]
    .apply(
        classificar_grupo_macro
    )
)


# ============================================================
# 11. CLASSIFICAÇÃO DA TRANSIÇÃO DE ESTADO
# ============================================================

def classificar_transicao(row):

    anterior = row["Grupo_Evento_Anterior"]
    atual = row["Grupo_Evento_Atual"]

    familia_anterior = row[
        "Familia_Evento_Anterior"
    ]

    familia_atual = row[
        "Familia_Evento_Atual"
    ]

    # --------------------------------------------------------
    # INFORMAÇÃO ANTERIOR AUSENTE
    # --------------------------------------------------------

    if anterior == "Desconhecido":

        if atual == "Alarme":
            return "Alarme sem estado anterior conhecido"

        if atual == "Pré-Alarme":
            return "Pré-Alarme sem estado anterior conhecido"

        if atual == "Comunicação / Confiabilidade":
            return "Falha comunicação sem estado anterior conhecido"

        return "Estado anterior desconhecido"

    # --------------------------------------------------------
    # ESTADO SEM ALTERAÇÃO
    # --------------------------------------------------------

    if familia_anterior == familia_atual:
        return "Estado mantido"

    # --------------------------------------------------------
    # NORMAL → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Pré-Alarme"
    ):
        return "Entrada em Pré-Alarme"

    # --------------------------------------------------------
    # NORMAL → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Alarme"
    ):
        return "Entrada em Alarme"

    # --------------------------------------------------------
    # PRÉ-ALARME → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Alarme"
    ):
        return "Escalada Pré-Alarme → Alarme"

    # --------------------------------------------------------
    # ALARME → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Pré-Alarme"
    ):
        return "Redução Alarme → Pré-Alarme"

    # --------------------------------------------------------
    # ALARME → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Normal"
    ):
        return "Normalização de Alarme"

    # --------------------------------------------------------
    # PRÉ-ALARME → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Normal"
    ):
        return "Normalização de Pré-Alarme"

    # --------------------------------------------------------
    # NORMAL → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Normal"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Entrada em Falha de Comunicação"

    # --------------------------------------------------------
    # ALARME → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Alarme → Falha de Comunicação"

    # --------------------------------------------------------
    # PRÉ-ALARME → FALHA COMUNICAÇÃO
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Comunicação / Confiabilidade"
    ):
        return "Pré-Alarme → Falha de Comunicação"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → NORMAL
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Normal"
    ):
        return "Restabelecimento de Comunicação"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → ALARME
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Alarme"
    ):
        return "Comunicação Restabelecida → Alarme"

    # --------------------------------------------------------
    # FALHA COMUNICAÇÃO → PRÉ-ALARME
    # --------------------------------------------------------

    if (
        anterior == "Comunicação / Confiabilidade"
        and atual == "Pré-Alarme"
    ):
        return "Comunicação Restabelecida → Pré-Alarme"

    # --------------------------------------------------------
    # TROCA ENTRE TIPOS DE ALARME
    # --------------------------------------------------------

    if (
        anterior == "Alarme"
        and atual == "Alarme"
    ):
        return "Mudança entre Alarmes"

    # --------------------------------------------------------
    # TROCA ENTRE PRÉ-ALARMES
    # --------------------------------------------------------

    if (
        anterior == "Pré-Alarme"
        and atual == "Pré-Alarme"
    ):
        return "Mudança entre Pré-Alarmes"

    # --------------------------------------------------------
    # EVENTOS DE SISTEMA
    # --------------------------------------------------------

    if atual == "Sistema":
        return "Evento de Sistema"

    if atual == "SDAI":
        return "Evento SDAI"

    # --------------------------------------------------------
    # OUTRAS TRANSIÇÕES
    # --------------------------------------------------------

    return (
        f"{anterior} → {atual}"
    )


print("\nClassificando transições de estado...")

df["Tipo_Transicao"] = (
    df.apply(
        classificar_transicao,
        axis=1
    )
)

print("Transições classificadas.")


# ============================================================
# 12. FLAGS OPERACIONAIS PRINCIPAIS
# ============================================================

df["Flag_Entrada_Alarme"] = (
    df["Tipo_Transicao"]
    .isin(
        [
            "Entrada em Alarme",
            "Escalada Pré-Alarme → Alarme",
            "Comunicação Restabelecida → Alarme",
            "Alarme sem estado anterior conhecido"
        ]
    )
)


df["Flag_Entrada_PreAlarme"] = (
    df["Tipo_Transicao"]
    .isin(
        [
            "Entrada em Pré-Alarme",
            "Comunicação Restabelecida → Pré-Alarme",
            "Pré-Alarme sem estado anterior conhecido"
        ]
    )
)


df["Flag_Normalizacao_Alarme"] = (
    df["Tipo_Transicao"]
    ==
    "Normalização de Alarme"
)


df["Flag_Normalizacao_PreAlarme"] = (
    df["Tipo_Transicao"]
    ==
    "Normalização de Pré-Alarme"
)


df["Flag_Falha_Comunicacao"] = (
    df["Grupo_Evento_Atual"]
    ==
    "Comunicação / Confiabilidade"
)


df["Flag_Restabelecimento_Comunicacao"] = (
    df["Tipo_Transicao"]
    ==
    "Restabelecimento de Comunicação"
)


df["Flag_Evento_Sistema"] = (
    df["Grupo_Evento_Atual"]
    ==
    "Sistema"
)


df["Flag_Evento_SDAI"] = (
    df["Grupo_Evento_Atual"]
    ==
    "SDAI"
)


# ============================================================
# 13. NATUREZA DO REGISTRO
# ============================================================

def classificar_natureza(row):

    if row["Flag_Entrada_Alarme"]:
        return "Geração de Alarme"

    if row["Flag_Entrada_PreAlarme"]:
        return "Geração de Pré-Alarme"

    if row["Flag_Normalizacao_Alarme"]:
        return "Normalização de Alarme"

    if row["Flag_Normalizacao_PreAlarme"]:
        return "Normalização de Pré-Alarme"

    if row["Flag_Restabelecimento_Comunicacao"]:
        return "Restabelecimento de Comunicação"

    if row["Flag_Falha_Comunicacao"]:
        return "Falha de Comunicação / Confiabilidade"

    if row["Flag_Evento_Sistema"]:
        return "Evento de Sistema"

    if row["Flag_Evento_SDAI"]:
        return "Evento SDAI"

    if row["Grupo_Evento_Atual"] == "Normal":
        return "Evento Normal"

    if row["Grupo_Evento_Atual"] == "Alarme":
        return "Alarme - outra transição"

    if row["Grupo_Evento_Atual"] == "Pré-Alarme":
        return "Pré-Alarme - outra transição"

    if row["Grupo_Evento_Atual"] == "Desconhecido":
        return "Não Classificado"

    return "Outro Evento"


df["Natureza_Evento"] = (
    df.apply(
        classificar_natureza,
        axis=1
    )
)


# ============================================================
# 14. FLAG DE EVENTO OPERACIONAL
# ============================================================

df["Flag_Evento_Operacional"] = (
    df["Natureza_Evento"]
    .isin(
        [
            "Geração de Alarme",
            "Geração de Pré-Alarme",
            "Normalização de Alarme",
            "Normalização de Pré-Alarme",
            "Falha de Comunicação / Confiabilidade",
            "Restabelecimento de Comunicação",
            "Alarme - outra transição",
            "Pré-Alarme - outra transição"
        ]
    )
)


# ============================================================
# 15. FLAG DE GERAÇÃO EFETIVA
# ============================================================

df["Flag_Geracao_Efetiva"] = (
    df["Natureza_Evento"]
    .isin(
        [
            "Geração de Alarme",
            "Geração de Pré-Alarme",
            "Falha de Comunicação / Confiabilidade"
        ]
    )
)


# ============================================================
# 16. DIREÇÃO DA TRANSIÇÃO
# ============================================================

def classificar_direcao(row):

    tipo = row["Tipo_Transicao"]

    if tipo in [
        "Entrada em Alarme",
        "Entrada em Pré-Alarme",
        "Escalada Pré-Alarme → Alarme",
        "Entrada em Falha de Comunicação"
    ]:
        return "Piora"

    if tipo in [
        "Normalização de Alarme",
        "Normalização de Pré-Alarme",
        "Restabelecimento de Comunicação",
        "Redução Alarme → Pré-Alarme"
    ]:
        return "Melhora"

    if tipo in [
        "Estado mantido",
        "Mudança entre Alarmes",
        "Mudança entre Pré-Alarmes"
    ]:
        return "Estável / Lateral"

    return "Indeterminado"


df["Direcao_Transicao"] = (
    df.apply(
        classificar_direcao,
        axis=1
    )
)


# ============================================================
# 17. SEVERIDADE LÓGICA DO ESTADO
# ============================================================

MAPA_SEVERIDADE = {
    "Normal": 0,
    "Pré-Alarme": 1,
    "Pré-Alarme Alto": 1,
    "Pré-Alarme Baixo": 1,
    "Alarme": 2,
    "Alarme Alto": 2,
    "Alarme Baixo": 2,
    "Offline": 3,
    "Unreliable": 3,
    "Sistema": np.nan,
    "SDAI": np.nan,
    "Outros": np.nan,
    "Desconhecido": np.nan
}


df["Nivel_Severidade_Anterior"] = (
    df["Familia_Evento_Anterior"]
    .map(
        MAPA_SEVERIDADE
    )
)


df["Nivel_Severidade_Atual"] = (
    df["Familia_Evento_Atual"]
    .map(
        MAPA_SEVERIDADE
    )
)


df["Variacao_Severidade"] = (
    df["Nivel_Severidade_Atual"]
    -
    df["Nivel_Severidade_Anterior"]
)


# ============================================================
# 18. IDENTIFICAÇÃO DE MUDANÇA REAL DE ESTADO
# ============================================================

df["Flag_Mudanca_Estado"] = (
    df["currentStatusEnumInfoId"]
    !=
    df["previousStatusEnumInfoId"]
)


# Cuida de NaNs
df["Flag_Mudanca_Estado"] = (
    df["Flag_Mudanca_Estado"]
    .fillna(False)
)


# ============================================================
# 19. RESUMO DAS FAMÍLIAS DE EVENTOS
# ============================================================

resumo_familias = (
    df["Familia_Evento_Atual"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_familias.columns = [
    "Familia_Evento",
    "Quantidade"
]

resumo_familias["Percentual"] = (
    resumo_familias["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 20. RESUMO DOS GRUPOS MACRO
# ============================================================

resumo_grupos = (
    df["Grupo_Evento_Atual"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_grupos.columns = [
    "Grupo_Evento",
    "Quantidade"
]

resumo_grupos["Percentual"] = (
    resumo_grupos["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 21. RESUMO DAS TRANSIÇÕES
# ============================================================

resumo_transicoes = (
    df["Tipo_Transicao"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_transicoes.columns = [
    "Tipo_Transicao",
    "Quantidade"
]

resumo_transicoes["Percentual"] = (
    resumo_transicoes["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 22. RESUMO DA NATUREZA DOS EVENTOS
# ============================================================

resumo_natureza = (
    df["Natureza_Evento"]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

resumo_natureza.columns = [
    "Natureza_Evento",
    "Quantidade"
]

resumo_natureza["Percentual"] = (
    resumo_natureza["Quantidade"]
    /
    len(df)
    *
    100
)


# ============================================================
# 23. MATRIZ ESTADO ANTERIOR × ESTADO ATUAL
# ============================================================

matriz_transicao = pd.crosstab(
    df["Familia_Evento_Anterior"],
    df["Familia_Evento_Atual"],
    margins=True,
    margins_name="Total"
)


# ============================================================
# 24. MATRIZ GRUPO ANTERIOR × GRUPO ATUAL
# ============================================================

matriz_grupo_transicao = pd.crosstab(
    df["Grupo_Evento_Anterior"],
    df["Grupo_Evento_Atual"],
    margins=True,
    margins_name="Total"
)


# ============================================================
# 25. CÓDIGOS SEM TRADUÇÃO DO ESTADO ANTERIOR
# ============================================================

codigos_anteriores_desconhecidos = (
    df.loc[
        df["Flag_Evento_Anterior_Desconhecido"],
        "previousStatusEnumInfoId"
    ]
    .value_counts(
        dropna=False
    )
    .reset_index()
)

codigos_anteriores_desconhecidos.columns = [
    "previousStatusEnumInfoId",
    "Quantidade"
]


# ============================================================
# 26. MAPA COMPLETO DOS CÓDIGOS EMPÍRICOS
# ============================================================

mapa_status_exportacao = (
    mapa_status_empirico_df
    .sort_values(
        [
            "currentStatusEnumInfoId",
            "Quantidade"
        ],
        ascending=[
            True,
            False
        ]
    )
    .copy()
)


# Percentual interno por código
mapa_status_exportacao[
    "Total_Codigo"
] = (
    mapa_status_exportacao
    .groupby(
        "currentStatusEnumInfoId"
    )["Quantidade"]
    .transform("sum")
)


mapa_status_exportacao[
    "Percentual_no_Codigo"
] = (
    mapa_status_exportacao[
        "Quantidade"
    ]
    /
    mapa_status_exportacao[
        "Total_Codigo"
    ]
    *
    100
)


# ============================================================
# 27. RESUMO MENSAL
# ============================================================

resumo_mensal = (
    df
    .groupby(
        [
            "Ano",
            "Mes_Numero",
            "Mes"
        ],
        observed=True
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Normalizacoes_PreAlarme=(
            "Flag_Normalizacao_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Restabelecimentos_Comunicacao=(
            "Flag_Restabelecimento_Comunicacao",
            "sum"
        ),

        Eventos_Sistema=(
            "Flag_Evento_Sistema",
            "sum"
        ),

        Eventos_Operacionais=(
            "Flag_Evento_Operacional",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
)


# ============================================================
# 28. RESUMO POR MANTENEDOR
# ============================================================

resumo_mantenedor = (
    df
    .groupby(
        "Mantenedor",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Eventos_Sistema=(
            "Flag_Evento_Sistema",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 29. RESUMO POR CATEGORIA
# ============================================================

resumo_categoria = (
    df
    .groupby(
        "Categoria",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 30. RESUMO POR TIPO
# ============================================================

resumo_tipo = (
    df
    .groupby(
        "Tipo",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 31. RESUMO POR TORRE
# ============================================================

resumo_torre = (
    df
    .groupby(
        "Torre",
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Equipamentos_Unicos=(
            "itemName",
            "nunique"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 32. RESUMO POR EQUIPAMENTO
# ============================================================

resumo_equipamento = (
    df
    .groupby(
        [
            "itemName",
            "itemDescription",
            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre"
        ],
        observed=True,
        dropna=False
    )
    .agg(
        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        )
    )
    .reset_index()
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


# ============================================================
# 33. KPIs DO MÓDULO 2
# ============================================================

total_eventos = len(df)

total_entrada_alarme = int(
    df["Flag_Entrada_Alarme"].sum()
)

total_entrada_prealarme = int(
    df["Flag_Entrada_PreAlarme"].sum()
)

total_normalizacao_alarme = int(
    df["Flag_Normalizacao_Alarme"].sum()
)

total_normalizacao_prealarme = int(
    df["Flag_Normalizacao_PreAlarme"].sum()
)

total_falha_comunicacao = int(
    df["Flag_Falha_Comunicacao"].sum()
)

total_restabelecimento = int(
    df[
        "Flag_Restabelecimento_Comunicacao"
    ].sum()
)

total_sistema = int(
    df["Flag_Evento_Sistema"].sum()
)

total_operacionais = int(
    df["Flag_Evento_Operacional"].sum()
)

total_geracoes_efetivas = int(
    df["Flag_Geracao_Efetiva"].sum()
)

eventos_anteriores_desconhecidos = int(
    df[
        "Flag_Evento_Anterior_Desconhecido"
    ].sum()
)


kpis_modulo2 = pd.DataFrame(
    {
        "Indicador": [
            "Registros totais",
            "Gerações de alarme",
            "Gerações de pré-alarme",
            "Normalizações de alarme",
            "Normalizações de pré-alarme",
            "Eventos em falha de comunicação",
            "Restabelecimentos de comunicação",
            "Eventos de sistema",
            "Eventos operacionais",
            "Gerações efetivas",
            "Estados anteriores sem tradução"
        ],

        "Quantidade": [
            total_eventos,
            total_entrada_alarme,
            total_entrada_prealarme,
            total_normalizacao_alarme,
            total_normalizacao_prealarme,
            total_falha_comunicacao,
            total_restabelecimento,
            total_sistema,
            total_operacionais,
            total_geracoes_efetivas,
            eventos_anteriores_desconhecidos
        ]
    }
)


kpis_modulo2["Percentual_Base"] = (
    kpis_modulo2["Quantidade"]
    /
    total_eventos
    *
    100
)


# ============================================================
# 34. EXIBIÇÃO DOS PRINCIPAIS RESULTADOS
# ============================================================

print("\n" + "=" * 90)
print("KPIs DO MÓDULO 2")
print("=" * 90)

display(
    kpis_modulo2
)


print("\n" + "=" * 90)
print("FAMÍLIAS DE EVENTOS")
print("=" * 90)

display(
    resumo_familias
)


print("\n" + "=" * 90)
print("TRANSIÇÕES DE ESTADO")
print("=" * 90)

display(
    resumo_transicoes.head(50)
)


print("\n" + "=" * 90)
print("NATUREZA DOS EVENTOS")
print("=" * 90)

display(
    resumo_natureza
)


print("\n" + "=" * 90)
print("RESUMO MENSAL")
print("=" * 90)

display(
    resumo_mensal
)


print("\n" + "=" * 90)
print("TOP 20 EQUIPAMENTOS POR GERAÇÕES EFETIVAS")
print("=" * 90)

display(
    resumo_equipamento.head(20)
)


# ============================================================
# 35. VALIDAÇÕES IMPORTANTES
# ============================================================

print("\n" + "=" * 90)
print("VALIDAÇÕES")
print("=" * 90)

print(
    f"Códigos de estado atuais identificados: "
    f"{df['currentStatusEnumInfoId'].nunique(dropna=True):,}"
)

print(
    f"Códigos anteriores identificados: "
    f"{df['previousStatusEnumInfoId'].nunique(dropna=True):,}"
)

print(
    f"Estados anteriores sem tradução: "
    f"{eventos_anteriores_desconhecidos:,}"
)

print(
    f"Percentual sem tradução: "
    f"{eventos_anteriores_desconhecidos / len(df) * 100:.4f}%"
)

print(
    f"Registros com mudança de estado: "
    f"{df['Flag_Mudanca_Estado'].sum():,}"
)

print(
    f"Registros sem mudança de estado: "
    f"{(~df['Flag_Mudanca_Estado']).sum():,}"
)


# ============================================================
# 36. OTIMIZAÇÃO DE MEMÓRIA
# ============================================================

colunas_category = [
    "Evento_Atual",
    "Evento_Anterior",
    "Familia_Evento_Atual",
    "Familia_Evento_Anterior",
    "Grupo_Evento_Atual",
    "Grupo_Evento_Anterior",
    "Tipo_Transicao",
    "Natureza_Evento",
    "Direcao_Transicao"
]


for coluna in colunas_category:

    if coluna in df.columns:

        df[coluna] = (
            df[coluna]
            .astype("category")
        )


# ============================================================
# 37. SALVAMENTO DA BASE CLASSIFICADA
# ============================================================

print("\nSalvando base classificada do Módulo 2...")


df.to_parquet(
    ARQUIVO_SAIDA,
    index=False
)


print(
    f"Base classificada criada:\n"
    f"{ARQUIVO_SAIDA}"
)


# ============================================================
# 38. EXPORTAÇÃO DO RELATÓRIO DO MÓDULO 2
# ============================================================

print("\nGerando relatório Excel do Módulo 2...")


with pd.ExcelWriter(
    ARQUIVO_RELATORIO,
    engine="openpyxl"
) as writer:

    kpis_modulo2.to_excel(
        writer,
        sheet_name="KPIs",
        index=False
    )

    mapa_status_exportacao.to_excel(
        writer,
        sheet_name="Mapa Codigos Status",
        index=False
    )

    codigos_anteriores_desconhecidos.to_excel(
        writer,
        sheet_name="Codigos Sem Traducao",
        index=False
    )

    resumo_familias.to_excel(
        writer,
        sheet_name="Familias Eventos",
        index=False
    )

    resumo_grupos.to_excel(
        writer,
        sheet_name="Grupos Eventos",
        index=False
    )

    resumo_transicoes.to_excel(
        writer,
        sheet_name="Transicoes",
        index=False
    )

    resumo_natureza.to_excel(
        writer,
        sheet_name="Natureza Eventos",
        index=False
    )

    matriz_transicao.to_excel(
        writer,
        sheet_name="Matriz Estados"
    )

    matriz_grupo_transicao.to_excel(
        writer,
        sheet_name="Matriz Grupos"
    )

    resumo_mensal.to_excel(
        writer,
        sheet_name="Resumo Mensal",
        index=False
    )

    resumo_mantenedor.to_excel(
        writer,
        sheet_name="Mantenedor",
        index=False
    )

    resumo_categoria.to_excel(
        writer,
        sheet_name="Categoria",
        index=False
    )

    resumo_tipo.to_excel(
        writer,
        sheet_name="Tipo",
        index=False
    )

    resumo_torre.to_excel(
        writer,
        sheet_name="Torre",
        index=False
    )

    resumo_equipamento.to_excel(
        writer,
        sheet_name="Equipamentos",
        index=False
    )


print(
    f"Relatório criado:\n"
    f"{ARQUIVO_RELATORIO}"
)


# ============================================================
# 39. AMOSTRA DA BASE CLASSIFICADA
# ============================================================

COLUNAS_AMOSTRA = [
    "DataHoraLocal",
    "priority",

    "previousStatusEnumInfoId",
    "currentStatusEnumInfoId",

    "Evento_Anterior",
    "Evento_Atual",

    "Familia_Evento_Anterior",
    "Familia_Evento_Atual",

    "Grupo_Evento_Anterior",
    "Grupo_Evento_Atual",

    "Tipo_Transicao",
    "Direcao_Transicao",
    "Natureza_Evento",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao",

    "itemName",
    "itemDescription",
    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre"
]


print("\n" + "=" * 90)
print("AMOSTRA DA BASE CLASSIFICADA")
print("=" * 90)

display(
    df[
        COLUNAS_AMOSTRA
    ].head(30)
)


# ============================================================
# 40. FINALIZAÇÃO
# ============================================================

print("\n" + "=" * 90)
print("MÓDULO 2 CONCLUÍDO COM SUCESSO")
print("=" * 90)

print("\nArquivos gerados:")

print(
    f"1. {ARQUIVO_SAIDA}"
)

print(
    f"2. {ARQUIVO_RELATORIO}"
)


# ============================================================
# 41. DOWNLOAD DO RELATÓRIO
# ============================================================

from google.colab import files

print(
    "\nIniciando download do relatório Excel..."
)

files.download(
    ARQUIVO_RELATORIO
)


print(
    "\nPara baixar também a base classificada em Parquet,"
    "\nexecute posteriormente:"
)

print(
    f'files.download("{ARQUIVO_SAIDA}")'
)

MÓDULO 2 - CLASSIFICAÇÃO INTELIGENTE E TRANSIÇÕES

Carregando base tratada do Módulo 1...
Base carregada com sucesso.
Registros: 1,413,882
Colunas iniciais: 34

Construindo dicionário empírico de códigos de estado...
Códigos mapeados empiricamente: 12

Classificando transições de estado...
Transições classificadas.

KPIs DO MÓDULO 2


,Indicador,Quantidade,Percentual_Base
0,Registros totais,1413882,100.00
1,Gerações de alarme,240277,16.99
2,Gerações de pré-alarme,296725,20.99
3,Normalizações de alarme,266494,18.85
4,Normalizações de pré-alarme,263966,18.67
5,Eventos em falha de comunicação,100564,7.11
6,Restabelecimentos de comunicação,8074,0.57
7,Eventos de sistema,7171,0.51
8,Eventos operacionais,1320425,93.39
9,Gerações efetivas,637566,45.09



FAMÍLIAS DE EVENTOS


,Familia_Evento,Quantidade,Percentual
0,Normal,624812,44.19
1,Pré-Alarme Baixo,196477,13.90
2,Pré-Alarme Alto,137370,9.72
3,Alarme,136278,9.64
4,Alarme Alto,116421,8.23
5,Alarme Baixo,94781,6.70
6,Offline,85864,6.07
7,Unreliable,14700,1.04
8,Sistema,7171,0.51
9,SDAI,8,0.00



TRANSIÇÕES DE ESTADO


...

,DataHoraLocal,priority,previousStatusEnumInfoId,currentStatusEnumInfoId,Evento_Anterior,Evento_Atual,Familia_Evento_Anterior,Familia_Evento_Atual,Grupo_Evento_Anterior,Grupo_Evento_Atual,Tipo_Transicao,Direcao_Transicao,Natureza_Evento,Flag_Entrada_Alarme,Flag_Entrada_PreAlarme,Flag_Normalizacao_Alarme,Flag_Normalizacao_PreAlarme,Flag_Falha_Comunicacao,Flag_Restabelecimento_Comunicacao,itemName,itemDescription,Categoria,Mantenedor,Tipo,Torre
0,2026-05-28 18:52:33-03:00,106,2,22,Normal,Off-line,Normal,Offline,Normal,Comunicação / Confiabilidade,Entrada em Falha de Comunicação,Piora,Falha de Comunicação / Confiabilidade,False,False,False,False,True,False,89-CGM09090 Default State,<NA>,Geral,Others,Off-line,TAE
1,2026-01-01 06:41:46-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
2,2026-01-01 07:31:48-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
3,2026-01-03 05:38:33-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
4,2026-01-03 06:28:33-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
5,2026-01-16 14:59:12-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
6,2026-01-16 15:49:08-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
7,2026-01-17 22:40:44-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
8,2026-01-17 23:30:40-03:00,200,22,2,Alarme,Normal,Alarme,Normal,Alarme,Normal,Normalização de Alarme,Melhora,Normalização de Alarme,False,False,True,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6
9,2026-01-18 00:58:33-03:00,70,2,22,Normal,Alarme,Normal,Alarme,Normal,Alarme,Entrada em Alarme,Piora,Geração de Alarme,True,False,False,False,False,False,Alarme_BB_TAE,<NA>,Hidráulica,Hidráulica,Other,Bloco E6



MÓDULO 2 CONCLUÍDO COM SUCESSO

Arquivos gerados:
1. Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
2. Relatorio_Modulo2_Classificacao_Alarmes_Metasys.xlsx

Iniciando download do relatório Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Para baixar também a base classificada em Parquet,
execute posteriormente:
files.download("Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet")


In [8]:
# ================================================================
# PROJETO: ANÁLISE DE ALARMES METASYS 2026
#
# MÓDULO 3
# ANÁLISE DE VOLUMETRIA DOS EVENTOS
#
# Base de entrada:
# Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet
#
# Objetivos:
# - Análise mensal
# - Análise diária
# - Análise horária
# - Dia da semana
# - Período do dia
# - Prioridade
# - Categoria
# - Mantenedor
# - Tipo
# - Torre
# - TTL Torre
# - Matrizes cruzadas
# - Rankings preliminares
# - Identificação de dias e horas de maior volumetria
# - Comparação Eventos Totais × Eventos Operacionais ×
#   Gerações Efetivas
# ================================================================


# ================================================================
# 1. BIBLIOTECAS
# ================================================================

import pandas as pd
import numpy as np

from pathlib import Path
from IPython.display import display

import warnings
warnings.filterwarnings("ignore")


pd.set_option(
    "display.max_columns",
    200
)

pd.set_option(
    "display.max_rows",
    200
)

pd.set_option(
    "display.width",
    250
)

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}"
)


# ================================================================
# 2. CONFIGURAÇÃO DOS ARQUIVOS
# ================================================================

ARQUIVO_ENTRADA = (
    "Base_Alarmes_Metasys_2026_Classificada_Modulo2.parquet"
)

ARQUIVO_RELATORIO = (
    "Relatorio_Modulo3_Volumetria_Alarmes_Metasys.xlsx"
)


print("=" * 100)
print("MÓDULO 3 - ANÁLISE DE VOLUMETRIA DOS ALARMES METASYS")
print("=" * 100)


# ================================================================
# 3. VERIFICAÇÃO / UPLOAD DO ARQUIVO
# ================================================================

if not Path(ARQUIVO_ENTRADA).exists():

    print(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "não foi localizado no ambiente atual."
    )

    print(
        "\nSelecione o arquivo Parquet gerado no Módulo 2."
    )

    from google.colab import files

    uploaded = files.upload()


if not Path(ARQUIVO_ENTRADA).exists():

    raise FileNotFoundError(
        f"\nO arquivo '{ARQUIVO_ENTRADA}' "
        "continua não disponível."
    )


# ================================================================
# 4. CARREGAMENTO DA BASE
# ================================================================

print("\nCarregando base do Módulo 2...")


df = pd.read_parquet(
    ARQUIVO_ENTRADA
)


print("Base carregada com sucesso.")

print(
    f"Registros: {len(df):,}"
)

print(
    f"Colunas: {df.shape[1]}"
)


# ================================================================
# 5. VERIFICAÇÃO DAS COLUNAS NECESSÁRIAS
# ================================================================

COLUNAS_NECESSARIAS = [

    "DataHoraLocal",

    "Ano",
    "Mes_Numero",
    "Mes",

    "Data",
    "Hora",

    "Dia_Semana_Numero",
    "Dia_Semana",
    "Periodo_Dia",

    "priority",

    "itemName",
    "itemDescription",

    "Categoria",
    "Mantenedor",
    "Tipo",
    "Torre",
    "TTL Torre",

    "Familia_Evento_Atual",
    "Grupo_Evento_Atual",
    "Natureza_Evento",

    "Flag_Entrada_Alarme",
    "Flag_Entrada_PreAlarme",

    "Flag_Normalizacao_Alarme",
    "Flag_Normalizacao_PreAlarme",

    "Flag_Falha_Comunicacao",
    "Flag_Restabelecimento_Comunicacao",

    "Flag_Evento_Sistema",

    "Flag_Evento_Operacional",
    "Flag_Geracao_Efetiva"
]


colunas_faltantes = [

    coluna

    for coluna
    in COLUNAS_NECESSARIAS

    if coluna not in df.columns
]


if colunas_faltantes:

    raise ValueError(

        "\nAs seguintes colunas necessárias "
        "não foram encontradas:\n\n"

        + "\n".join(
            colunas_faltantes
        )
    )


# ================================================================
# 6. AJUSTES DE TIPOS
# ================================================================

df["DataHoraLocal"] = pd.to_datetime(
    df["DataHoraLocal"],
    errors="coerce"
)


# Cria uma data pandas sem timezone para agrupamento e Excel
df["Data_Analise"] = (
    df["DataHoraLocal"]
    .dt.tz_localize(None)
    .dt.normalize()
)


df["AnoMes"] = (
    df["DataHoraLocal"]
    .dt.strftime("%Y-%m")
)


# Hora cheia
df["DataHora_Hora"] = (
    df["DataHoraLocal"]
    .dt.tz_localize(None)
    .dt.floor("h")
)


# ================================================================
# 7. ORDEM DOS MESES / DIAS
# ================================================================

ORDEM_MESES = [

    "Janeiro",
    "Fevereiro",
    "Março",
    "Abril",
    "Maio",
    "Junho",
    "Julho",
    "Agosto",
    "Setembro",
    "Outubro",
    "Novembro",
    "Dezembro"
]


ORDEM_DIAS = [

    "Segunda",
    "Terça",
    "Quarta",
    "Quinta",
    "Sexta",
    "Sábado",
    "Domingo"
]


ORDEM_PERIODOS = [

    "Madrugada",
    "Manhã",
    "Tarde",
    "Noite"
]


# ================================================================
# 8. FUNÇÃO BASE DE AGREGAÇÃO
# ================================================================

def gerar_resumo(df_base, dimensoes):

    """
    Gera os principais indicadores de volumetria
    para uma ou mais dimensões.
    """

    resultado = (

        df_base
        .groupby(
            dimensoes,
            observed=True,
            dropna=False
        )
        .agg(

            Eventos_Totais=(
                "itemName",
                "size"
            ),

            Eventos_Operacionais=(
                "Flag_Evento_Operacional",
                "sum"
            ),

            Geracoes_Efetivas=(
                "Flag_Geracao_Efetiva",
                "sum"
            ),

            Geracoes_Alarme=(
                "Flag_Entrada_Alarme",
                "sum"
            ),

            Geracoes_PreAlarme=(
                "Flag_Entrada_PreAlarme",
                "sum"
            ),

            Falhas_Comunicacao=(
                "Flag_Falha_Comunicacao",
                "sum"
            ),

            Normalizacoes_Alarme=(
                "Flag_Normalizacao_Alarme",
                "sum"
            ),

            Normalizacoes_PreAlarme=(
                "Flag_Normalizacao_PreAlarme",
                "sum"
            ),

            Restabelecimentos_Comunicacao=(
                "Flag_Restabelecimento_Comunicacao",
                "sum"
            ),

            Eventos_Sistema=(
                "Flag_Evento_Sistema",
                "sum"
            ),

            Equipamentos_Unicos=(
                "itemName",
                "nunique"
            )
        )

        .reset_index()
    )


    # ------------------------------------------------------------
    # Percentual de gerações efetivas sobre eventos totais
    # ------------------------------------------------------------

    resultado[
        "Perc_Geracoes_Efetivas"
    ] = np.where(

        resultado[
            "Eventos_Totais"
        ] > 0,

        (
            resultado[
                "Geracoes_Efetivas"
            ]
            /
            resultado[
                "Eventos_Totais"
            ]
            *
            100
        ),

        0
    )


    # ------------------------------------------------------------
    # Percentual operacional
    # ------------------------------------------------------------

    resultado[
        "Perc_Eventos_Operacionais"
    ] = np.where(

        resultado[
            "Eventos_Totais"
        ] > 0,

        (
            resultado[
                "Eventos_Operacionais"
            ]
            /
            resultado[
                "Eventos_Totais"
            ]
            *
            100
        ),

        0
    )


    return resultado


# ================================================================
# 9. KPIs GERAIS
# ================================================================

TOTAL_EVENTOS = len(df)

TOTAL_OPERACIONAIS = int(
    df[
        "Flag_Evento_Operacional"
    ].sum()
)

TOTAL_GERACOES = int(
    df[
        "Flag_Geracao_Efetiva"
    ].sum()
)

TOTAL_ALARMES = int(
    df[
        "Flag_Entrada_Alarme"
    ].sum()
)

TOTAL_PREALARMES = int(
    df[
        "Flag_Entrada_PreAlarme"
    ].sum()
)

TOTAL_COMUNICACAO = int(
    df[
        "Flag_Falha_Comunicacao"
    ].sum()
)

TOTAL_NORMALIZACOES_ALARME = int(
    df[
        "Flag_Normalizacao_Alarme"
    ].sum()
)

TOTAL_NORMALIZACOES_PREALARME = int(
    df[
        "Flag_Normalizacao_PreAlarme"
    ].sum()
)

TOTAL_EQUIPAMENTOS = (
    df[
        "itemName"
    ].nunique()
)

TOTAL_DIAS = (
    df[
        "Data_Analise"
    ].nunique()
)


kpis_gerais = pd.DataFrame({

    "Indicador": [

        "Eventos totais",
        "Eventos operacionais",
        "Gerações efetivas",

        "Gerações de alarme",
        "Gerações de pré-alarme",
        "Falhas de comunicação",

        "Normalizações de alarme",
        "Normalizações de pré-alarme",

        "Equipamentos únicos",
        "Dias analisados"
    ],

    "Quantidade": [

        TOTAL_EVENTOS,
        TOTAL_OPERACIONAIS,
        TOTAL_GERACOES,

        TOTAL_ALARMES,
        TOTAL_PREALARMES,
        TOTAL_COMUNICACAO,

        TOTAL_NORMALIZACOES_ALARME,
        TOTAL_NORMALIZACOES_PREALARME,

        TOTAL_EQUIPAMENTOS,
        TOTAL_DIAS
    ]

})


# Percentual somente para indicadores que representam registros
kpis_gerais["Percentual_Base"] = np.nan


mascara_percentual = (

    kpis_gerais[
        "Indicador"
    ].isin([

        "Eventos totais",
        "Eventos operacionais",
        "Gerações efetivas",

        "Gerações de alarme",
        "Gerações de pré-alarme",
        "Falhas de comunicação",

        "Normalizações de alarme",
        "Normalizações de pré-alarme"
    ])
)


kpis_gerais.loc[
    mascara_percentual,
    "Percentual_Base"
] = (

    kpis_gerais.loc[
        mascara_percentual,
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 10. MÉDIAS GERAIS
# ================================================================

media_eventos_dia = (
    TOTAL_EVENTOS
    /
    TOTAL_DIAS
)

media_geracoes_dia = (
    TOTAL_GERACOES
    /
    TOTAL_DIAS
)

media_alarm_dia = (
    TOTAL_ALARMES
    /
    TOTAL_DIAS
)

media_prealarme_dia = (
    TOTAL_PREALARMES
    /
    TOTAL_DIAS
)


medias_gerais = pd.DataFrame({

    "Indicador": [

        "Média de eventos totais por dia",
        "Média de gerações efetivas por dia",
        "Média de gerações de alarme por dia",
        "Média de gerações de pré-alarme por dia"
    ],

    "Valor": [

        media_eventos_dia,
        media_geracoes_dia,
        media_alarm_dia,
        media_prealarme_dia
    ]

})


# ================================================================
# 11. VOLUMETRIA MENSAL
# ================================================================

resumo_mensal = gerar_resumo(

    df,

    [
        "Ano",
        "Mes_Numero",
        "Mes"
    ]
)


resumo_mensal = (

    resumo_mensal
    .sort_values(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
    .reset_index(
        drop=True
    )
)


# Número de dias existentes em cada mês
dias_mes = (

    df
    .groupby(
        [
            "Ano",
            "Mes_Numero"
        ],
        observed=True
    )[
        "Data_Analise"
    ]
    .nunique()
    .reset_index(
        name="Dias_Analisados"
    )
)


resumo_mensal = resumo_mensal.merge(

    dias_mes,

    on=[
        "Ano",
        "Mes_Numero"
    ],

    how="left"
)


# Médias por dia
resumo_mensal[
    "Media_Eventos_Dia"
] = (

    resumo_mensal[
        "Eventos_Totais"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_Geracoes_Dia"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_Alarmes_Dia"
] = (

    resumo_mensal[
        "Geracoes_Alarme"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


resumo_mensal[
    "Media_PreAlarmes_Dia"
] = (

    resumo_mensal[
        "Geracoes_PreAlarme"
    ]
    /
    resumo_mensal[
        "Dias_Analisados"
    ]
)


# ================================================================
# 12. VARIAÇÃO MÊS CONTRA MÊS
# ================================================================

resumo_mensal[
    "Var_MoM_Eventos_Perc"
] = (

    resumo_mensal[
        "Eventos_Totais"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_Geracoes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_Alarmes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Alarme"
    ]
    .pct_change()
    *
    100
)


resumo_mensal[
    "Var_MoM_PreAlarmes_Perc"
] = (

    resumo_mensal[
        "Geracoes_PreAlarme"
    ]
    .pct_change()
    *
    100
)


# ================================================================
# 13. VARIAÇÃO CONTRA O PRIMEIRO MÊS
# ================================================================

if len(resumo_mensal) > 0:

    base_eventos = (
        resumo_mensal.iloc[0][
            "Eventos_Totais"
        ]
    )

    base_geracoes = (
        resumo_mensal.iloc[0][
            "Geracoes_Efetivas"
        ]
    )

    base_alarm = (
        resumo_mensal.iloc[0][
            "Geracoes_Alarme"
        ]
    )

    base_prealarme = (
        resumo_mensal.iloc[0][
            "Geracoes_PreAlarme"
        ]
    )


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Eventos_Perc"
    ] = (

        (
            resumo_mensal[
                "Eventos_Totais"
            ]
            /
            base_eventos
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Geracoes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_Efetivas"
            ]
            /
            base_geracoes
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_Alarmes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_Alarme"
            ]
            /
            base_alarm
        )
        -
        1
    ) * 100


    resumo_mensal[
        "Var_vs_Primeiro_Mes_PreAlarmes_Perc"
    ] = (

        (
            resumo_mensal[
                "Geracoes_PreAlarme"
            ]
            /
            base_prealarme
        )
        -
        1
    ) * 100


# ================================================================
# 14. PARTICIPAÇÃO MENSAL
# ================================================================

resumo_mensal[
    "Participacao_Geracoes_Perc"
] = (

    resumo_mensal[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 15. VOLUMETRIA DIÁRIA
# ================================================================

resumo_diario = gerar_resumo(

    df,

    [
        "Data_Analise"
    ]
)


resumo_diario = (

    resumo_diario
    .sort_values(
        "Data_Analise"
    )
    .reset_index(
        drop=True
    )
)


# Acrescenta informações temporais
resumo_diario[
    "Ano"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.year
)


resumo_diario[
    "Mes_Numero"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.month
)


resumo_diario[
    "Mes"
] = (
    resumo_diario[
        "Mes_Numero"
    ]
    .map({

        1: "Janeiro",
        2: "Fevereiro",
        3: "Março",
        4: "Abril",
        5: "Maio",
        6: "Junho",
        7: "Julho",
        8: "Agosto",
        9: "Setembro",
        10: "Outubro",
        11: "Novembro",
        12: "Dezembro"
    })
)


resumo_diario[
    "Dia_Semana_Numero"
] = (
    resumo_diario[
        "Data_Analise"
    ]
    .dt.dayofweek
    +
    1
)


resumo_diario[
    "Dia_Semana"
] = (
    resumo_diario[
        "Dia_Semana_Numero"
    ]
    .map({

        1: "Segunda",
        2: "Terça",
        3: "Quarta",
        4: "Quinta",
        5: "Sexta",
        6: "Sábado",
        7: "Domingo"
    })
)


# ================================================================
# 16. TOP DIAS POR GERAÇÕES EFETIVAS
# ================================================================

top_dias = (

    resumo_diario
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .head(50)
    .reset_index(
        drop=True
    )
)


# ================================================================
# 17. VOLUMETRIA HORÁRIA - HORA DO DIA
# ================================================================

resumo_hora = gerar_resumo(

    df,

    [
        "Hora"
    ]
)


resumo_hora = (

    resumo_hora
    .sort_values(
        "Hora"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 18. VOLUMETRIA POR HORA CRONOLÓGICA
# ================================================================

resumo_datahora = gerar_resumo(

    df,

    [
        "DataHora_Hora"
    ]
)


resumo_datahora = (

    resumo_datahora
    .sort_values(
        "DataHora_Hora"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 19. TOP HORAS DE MAIOR VOLUMETRIA
# ================================================================

top_horas = (

    resumo_datahora
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .head(100)
    .reset_index(
        drop=True
    )
)


# ================================================================
# 20. DIA DA SEMANA
# ================================================================

resumo_dia_semana = gerar_resumo(

    df,

    [
        "Dia_Semana_Numero",
        "Dia_Semana"
    ]
)


resumo_dia_semana = (

    resumo_dia_semana
    .sort_values(
        "Dia_Semana_Numero"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 21. MÉDIA POR OCORRÊNCIA DO DIA DA SEMANA
# ================================================================

dias_semana_contagem = (

    df[
        [
            "Data_Analise",
            "Dia_Semana_Numero"
        ]
    ]
    .drop_duplicates()
    .groupby(
        "Dia_Semana_Numero"
    )
    .size()
    .reset_index(
        name="Qtd_Dias"
    )
)


resumo_dia_semana = resumo_dia_semana.merge(

    dias_semana_contagem,

    on="Dia_Semana_Numero",

    how="left"
)


resumo_dia_semana[
    "Media_Geracoes_por_Dia"
] = (

    resumo_dia_semana[
        "Geracoes_Efetivas"
    ]
    /
    resumo_dia_semana[
        "Qtd_Dias"
    ]
)


resumo_dia_semana[
    "Media_Alarmes_por_Dia"
] = (

    resumo_dia_semana[
        "Geracoes_Alarme"
    ]
    /
    resumo_dia_semana[
        "Qtd_Dias"
    ]
)


# ================================================================
# 22. PERÍODO DO DIA
# ================================================================

resumo_periodo = gerar_resumo(

    df,

    [
        "Periodo_Dia"
    ]
)


ordem_periodo_map = {

    "Madrugada": 1,
    "Manhã": 2,
    "Tarde": 3,
    "Noite": 4
}


resumo_periodo[
    "Ordem"
] = (

    resumo_periodo[
        "Periodo_Dia"
    ]
    .astype(str)
    .map(
        ordem_periodo_map
    )
)


resumo_periodo = (

    resumo_periodo
    .sort_values(
        "Ordem"
    )
    .drop(
        columns="Ordem"
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 23. PRIORIDADE
# ================================================================

resumo_prioridade = gerar_resumo(

    df,

    [
        "priority"
    ]
)


resumo_prioridade = (

    resumo_prioridade
    .sort_values(
        "priority",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)


# Participação das gerações efetivas
resumo_prioridade[
    "Participacao_Geracoes_Perc"
] = (

    resumo_prioridade[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 24. CATEGORIA
# ================================================================

resumo_categoria = gerar_resumo(

    df,

    [
        "Categoria"
    ]
)


resumo_categoria = (

    resumo_categoria
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_categoria[
    "Participacao_Geracoes_Perc"
] = (

    resumo_categoria[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_categoria[
    "Participacao_Acumulada_Perc"
] = (

    resumo_categoria[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# ================================================================
# 25. MANTENEDOR
# ================================================================

resumo_mantenedor = gerar_resumo(

    df,

    [
        "Mantenedor"
    ]
)


resumo_mantenedor = (

    resumo_mantenedor
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_mantenedor[
    "Participacao_Geracoes_Perc"
] = (

    resumo_mantenedor[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 26. TIPO
# ================================================================

resumo_tipo = gerar_resumo(

    df,

    [
        "Tipo"
    ]
)


resumo_tipo = (

    resumo_tipo
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_tipo[
    "Participacao_Geracoes_Perc"
] = (

    resumo_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_tipo[
    "Participacao_Acumulada_Perc"
] = (

    resumo_tipo[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# ================================================================
# 27. TORRE
# ================================================================

resumo_torre = gerar_resumo(

    df,

    [
        "Torre"
    ]
)


resumo_torre = (

    resumo_torre
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


resumo_torre[
    "Participacao_Geracoes_Perc"
] = (

    resumo_torre[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 28. TTL TORRE
# ================================================================

resumo_ttl_torre = gerar_resumo(

    df,

    [
        "TTL Torre"
    ]
)


resumo_ttl_torre = (

    resumo_ttl_torre
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


# ================================================================
# 29. MATRIZ MANTENEDOR × TIPO
# ================================================================

matriz_mantenedor_tipo = pd.pivot_table(

    df,

    index="Mantenedor",

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 30. MATRIZ CATEGORIA × TIPO
# ================================================================

matriz_categoria_tipo = pd.pivot_table(

    df,

    index="Categoria",

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 31. MATRIZ TORRE × CATEGORIA
# ================================================================

matriz_torre_categoria = pd.pivot_table(

    df,

    index="Torre",

    columns="Categoria",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 32. MATRIZ MÊS × MANTENEDOR
# ================================================================

matriz_mes_mantenedor = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Mantenedor",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 33. MATRIZ MÊS × CATEGORIA
# ================================================================

matriz_mes_categoria = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Categoria",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 34. MATRIZ MÊS × TIPO
# ================================================================

matriz_mes_tipo = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Tipo",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 35. MATRIZ DIA DA SEMANA × HORA
# ================================================================

matriz_dia_hora = pd.pivot_table(

    df,

    index=[
        "Dia_Semana_Numero",
        "Dia_Semana"
    ],

    columns="Hora",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 36. MATRIZ MÊS × HORA
# ================================================================

matriz_mes_hora = pd.pivot_table(

    df,

    index=[
        "Mes_Numero",
        "Mes"
    ],

    columns="Hora",

    values="Flag_Geracao_Efetiva",

    aggfunc="sum",

    fill_value=0,

    observed=True,

    margins=True,

    margins_name="Total"
)


# ================================================================
# 37. RANKING PRELIMINAR DOS EQUIPAMENTOS
# ================================================================

resumo_equipamentos = (

    df
    .groupby(

        [
            "itemName",
            "itemDescription",
            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre",
            "TTL Torre"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Eventos_Totais=(
            "itemName",
            "size"
        ),

        Eventos_Operacionais=(
            "Flag_Evento_Operacional",
            "sum"
        ),

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        ),

        Normalizacoes_Alarme=(
            "Flag_Normalizacao_Alarme",
            "sum"
        ),

        Normalizacoes_PreAlarme=(
            "Flag_Normalizacao_PreAlarme",
            "sum"
        ),

        Prioridade_Minima=(
            "priority",
            "min"
        ),

        Prioridade_Mediana=(
            "priority",
            "median"
        ),

        Prioridade_Media=(
            "priority",
            "mean"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


resumo_equipamentos[
    "Participacao_Geracoes_Perc"
] = (

    resumo_equipamentos[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


resumo_equipamentos[
    "Participacao_Acumulada_Perc"
] = (

    resumo_equipamentos[
        "Participacao_Geracoes_Perc"
    ]
    .cumsum()
)


# Top 100 preliminar
top100_equipamentos = (

    resumo_equipamentos
    .head(100)
    .copy()
)


# ================================================================
# 38. RANKING POR MÊS E EQUIPAMENTO
# ================================================================

ranking_mensal_equip = (

    df
    .groupby(

        [
            "Ano",
            "Mes_Numero",
            "Mes",

            "itemName",
            "itemDescription",

            "Categoria",
            "Mantenedor",
            "Tipo",
            "Torre"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()
)


ranking_mensal_equip = (

    ranking_mensal_equip[

        ranking_mensal_equip[
            "Geracoes_Efetivas"
        ] > 0

    ]

    .sort_values(

        [
            "Ano",
            "Mes_Numero",
            "Geracoes_Efetivas"
        ],

        ascending=[
            True,
            True,
            False
        ]
    )
)


ranking_mensal_equip[
    "Ranking_Mes"
] = (

    ranking_mensal_equip
    .groupby(
        [
            "Ano",
            "Mes_Numero"
        ]
    )
    .cumcount()
    +
    1
)


top20_mensal = (

    ranking_mensal_equip[
        ranking_mensal_equip[
            "Ranking_Mes"
        ] <= 20
    ]
    .copy()
)


# ================================================================
# 39. VOLUMETRIA POR NATUREZA DO EVENTO
# ================================================================

resumo_natureza = (

    df[
        "Natureza_Evento"
    ]

    .value_counts(
        dropna=False
    )

    .reset_index()
)


resumo_natureza.columns = [

    "Natureza_Evento",
    "Quantidade"
]


resumo_natureza[
    "Percentual"
] = (

    resumo_natureza[
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 40. VOLUMETRIA POR FAMÍLIA
# ================================================================

resumo_familia = (

    df[
        "Familia_Evento_Atual"
    ]

    .value_counts(
        dropna=False
    )

    .reset_index()
)


resumo_familia.columns = [

    "Familia_Evento",
    "Quantidade"
]


resumo_familia[
    "Percentual"
] = (

    resumo_familia[
        "Quantidade"
    ]
    /
    TOTAL_EVENTOS
    *
    100
)


# ================================================================
# 41. TOP COMBINAÇÕES CATEGORIA × TIPO
# ================================================================

ranking_categoria_tipo = (

    df
    .groupby(

        [
            "Categoria",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_categoria_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_categoria_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 42. TOP COMBINAÇÕES MANTENEDOR × TIPO
# ================================================================

ranking_mantenedor_tipo = (

    df
    .groupby(

        [
            "Mantenedor",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_mantenedor_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_mantenedor_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 43. TOP COMBINAÇÕES TORRE × TIPO
# ================================================================

ranking_torre_tipo = (

    df
    .groupby(

        [
            "Torre",
            "Tipo"
        ],

        observed=True,

        dropna=False
    )

    .agg(

        Geracoes_Efetivas=(
            "Flag_Geracao_Efetiva",
            "sum"
        ),

        Geracoes_Alarme=(
            "Flag_Entrada_Alarme",
            "sum"
        ),

        Geracoes_PreAlarme=(
            "Flag_Entrada_PreAlarme",
            "sum"
        ),

        Falhas_Comunicacao=(
            "Flag_Falha_Comunicacao",
            "sum"
        )
    )

    .reset_index()

    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
)


ranking_torre_tipo[
    "Participacao_Geracoes_Perc"
] = (

    ranking_torre_tipo[
        "Geracoes_Efetivas"
    ]
    /
    TOTAL_GERACOES
    *
    100
)


# ================================================================
# 44. RESUMO DOS PONTOS PRINCIPAIS
# ================================================================

print("\n" + "=" * 100)
print("KPIs GERAIS")
print("=" * 100)

display(
    kpis_gerais
)


print("\n" + "=" * 100)
print("MÉDIAS GERAIS")
print("=" * 100)

display(
    medias_gerais
)


print("\n" + "=" * 100)
print("VOLUMETRIA MENSAL")
print("=" * 100)

display(
    resumo_mensal
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR CATEGORIA")
print("=" * 100)

display(
    resumo_categoria
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR MANTENEDOR")
print("=" * 100)

display(
    resumo_mantenedor
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR TIPO")
print("=" * 100)

display(
    resumo_tipo.head(30)
)


print("\n" + "=" * 100)
print("VOLUMETRIA POR TORRE")
print("=" * 100)

display(
    resumo_torre
)


print("\n" + "=" * 100)
print("TOP 20 EQUIPAMENTOS")
print("=" * 100)

display(
    resumo_equipamentos.head(20)
)


print("\n" + "=" * 100)
print("TOP 20 DIAS")
print("=" * 100)

display(
    top_dias.head(20)
)


print("\n" + "=" * 100)
print("TOP 20 HORAS")
print("=" * 100)

display(
    top_horas.head(20)
)


# ================================================================
# 45. CRIAÇÃO DE RESUMO EXECUTIVO
# ================================================================

mes_maior_geracao = (
    resumo_mensal
    .sort_values(
        "Geracoes_Efetivas",
        ascending=False
    )
    .iloc[0]
)

mes_menor_geracao = (
    resumo_mensal
    .sort_values(
        "Geracoes_Efetivas",
        ascending=True
    )
    .iloc[0]
)

dia_maior_geracao = top_dias.iloc[0]

hora_maior_geracao = top_horas.iloc[0]

categoria_maior = resumo_categoria.iloc[0]

mantenedor_maior = resumo_mantenedor.iloc[0]

tipo_maior = resumo_tipo.iloc[0]

torre_maior = resumo_torre.iloc[0]

equip_maior = resumo_equipamentos.iloc[0]


resumo_executivo = pd.DataFrame(
    {
        "Indicador": [
            "Total de registros",
            "Total de gerações efetivas",
            "% gerações efetivas / registros",
            "Média de gerações por dia",
            "Mês com maior geração",
            "Quantidade no mês de maior geração",
            "Mês com menor geração",
            "Quantidade no mês de menor geração",
            "Dia com maior geração",
            "Quantidade no dia de maior geração",
            "Hora cronológica de maior geração",
            "Quantidade na hora de maior geração",
            "Categoria com maior geração",
            "Mantenedor com maior geração",
            "Tipo com maior geração",
            "Torre com maior geração",
            "Equipamento com maior geração",
            "Gerações do equipamento líder"
        ],

        "Resultado": [
            f"{TOTAL_EVENTOS:,}",
            f"{TOTAL_GERACOES:,}",
            f"{TOTAL_GERACOES / TOTAL_EVENTOS * 100:.2f}%",
            f"{media_geracoes_dia:,.2f}",

            str(
                mes_maior_geracao["Mes"]
            ),

            f"{int(mes_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                mes_menor_geracao["Mes"]
            ),

            f"{int(mes_menor_geracao['Geracoes_Efetivas']):,}",

            str(
                dia_maior_geracao["Data_Analise"].date()
            ),

            f"{int(dia_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                hora_maior_geracao["DataHora_Hora"]
            ),

            f"{int(hora_maior_geracao['Geracoes_Efetivas']):,}",

            str(
                categoria_maior["Categoria"]
            ),

            str(
                mantenedor_maior["Mantenedor"]
            ),

            str(
                tipo_maior["Tipo"]
            ),

            str(
                torre_maior["Torre"]
            ),

            str(
                equip_maior["itemName"]
            ),

            f"{int(equip_maior['Geracoes_Efetivas']):,}"
        ]
    }
)


# ================================================================
# 46. EXPORTAÇÃO PARA EXCEL
# ================================================================

print("\nGerando relatório Excel do Módulo 3...")


with pd.ExcelWriter(

    ARQUIVO_RELATORIO,

    engine="openpyxl"

) as writer:


    resumo_executivo.to_excel(

        writer,

        sheet_name="Resumo Executivo",

        index=False
    )


    kpis_gerais.to_excel(

        writer,

        sheet_name="KPIs Gerais",

        index=False
    )


    medias_gerais.to_excel(

        writer,

        sheet_name="Medias Gerais",

        index=False
    )


    resumo_mensal.to_excel(

        writer,

        sheet_name="Mensal",

        index=False
    )


    resumo_diario.to_excel(

        writer,

        sheet_name="Diario",

        index=False
    )


    top_dias.to_excel(

        writer,

        sheet_name="Top Dias",

        index=False
    )


    resumo_hora.to_excel(

        writer,

        sheet_name="Hora Dia",

        index=False
    )


    resumo_datahora.to_excel(

        writer,

        sheet_name="Hora Cronologica",

        index=False
    )


    top_horas.to_excel(

        writer,

        sheet_name="Top Horas",

        index=False
    )


    resumo_dia_semana.to_excel(

        writer,

        sheet_name="Dia Semana",

        index=False
    )


    resumo_periodo.to_excel(

        writer,

        sheet_name="Periodo Dia",

        index=False
    )


    resumo_prioridade.to_excel(

        writer,

        sheet_name="Prioridades",

        index=False
    )


    resumo_categoria.to_excel(

        writer,

        sheet_name="Categorias",

        index=False
    )


    resumo_mantenedor.to_excel(

        writer,

        sheet_name="Mantenedores",

        index=False
    )


    resumo_tipo.to_excel(

        writer,

        sheet_name="Tipos",

        index=False
    )


    resumo_torre.to_excel(

        writer,

        sheet_name="Torres",

        index=False
    )


    resumo_ttl_torre.to_excel(

        writer,

        sheet_name="TTL Torre",

        index=False
    )


    resumo_natureza.to_excel(

        writer,

        sheet_name="Natureza Eventos",

        index=False
    )


    resumo_familia.to_excel(

        writer,

        sheet_name="Familia Eventos",

        index=False
    )


    top100_equipamentos.to_excel(

        writer,

        sheet_name="Top100 Equipamentos",

        index=False
    )


    top20_mensal.to_excel(

        writer,

        sheet_name="Top20 Mensal",

        index=False
    )


    ranking_categoria_tipo.to_excel(

        writer,

        sheet_name="Categoria x Tipo",

        index=False
    )


    ranking_mantenedor_tipo.to_excel(

        writer,

        sheet_name="Mantenedor x Tipo",

        index=False
    )


    ranking_torre_tipo.to_excel(

        writer,

        sheet_name="Torre x Tipo",

        index=False
    )


    matriz_mantenedor_tipo.to_excel(

        writer,

        sheet_name="Matriz Mant x Tipo"
    )


    matriz_categoria_tipo.to_excel(

        writer,

        sheet_name="Matriz Cat x Tipo"
    )


    matriz_torre_categoria.to_excel(

        writer,

        sheet_name="Matriz Torre x Cat"
    )


    matriz_mes_mantenedor.to_excel(

        writer,

        sheet_name="Mes x Mantenedor"
    )


    matriz_mes_categoria.to_excel(

        writer,

        sheet_name="Mes x Categoria"
    )


    matriz_mes_tipo.to_excel(

        writer,

        sheet_name="Mes x Tipo"
    )


    matriz_dia_hora.to_excel(

        writer,

        sheet_name="Dia Semana x Hora"
    )


    matriz_mes_hora.to_excel(

        writer,

        sheet_name="Mes x Hora"
    )


# ================================================================
# 47. FORMATAÇÃO BÁSICA DO EXCEL
# ================================================================

from openpyxl import load_workbook

from openpyxl.styles import (
    Font,
    PatternFill,
    Alignment,
    Border,
    Side
)

from openpyxl.utils import get_column_letter


wb = load_workbook(
    ARQUIVO_RELATORIO
)


thin_border = Border(

    bottom=Side(
        style="thin"
    )
)


for ws in wb.worksheets:

    # Congela cabeçalho
    ws.freeze_panes = "A2"

    # Filtro automático
    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    # Cabeçalho
    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

        cell.border = thin_border


    # Ajuste de largura
    for column_cells in ws.columns:

        max_length = 0

        coluna = get_column_letter(
            column_cells[0].column
        )

        for cell in column_cells:

            try:

                valor = (
                    ""
                    if cell.value is None
                    else str(cell.value)
                )

                if len(valor) > max_length:
                    max_length = len(valor)

            except:
                pass


        largura = min(
            max(
                max_length + 2,
                12
            ),
            45
        )

        ws.column_dimensions[
            coluna
        ].width = largura


wb.save(
    ARQUIVO_RELATORIO
)


# ================================================================
# 48. VALIDAÇÃO FINAL
# ================================================================

print("\n" + "=" * 100)
print("VALIDAÇÃO DOS TOTAIS")
print("=" * 100)


print(
    f"Eventos totais na base................: "
    f"{TOTAL_EVENTOS:,}"
)


print(
    f"Gerações efetivas na base.............: "
    f"{TOTAL_GERACOES:,}"
)


print(
    f"Soma das gerações no resumo mensal....: "
    f"{int(resumo_mensal['Geracoes_Efetivas'].sum()):,}"
)


print(
    f"Soma das gerações no resumo diário....: "
    f"{int(resumo_diario['Geracoes_Efetivas'].sum()):,}"
)


print(
    f"Soma das gerações por hora do dia.....: "
    f"{int(resumo_hora['Geracoes_Efetivas'].sum()):,}"
)


# Verificação
validacao_mensal = (
    int(
        resumo_mensal[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


validacao_diaria = (
    int(
        resumo_diario[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


validacao_hora = (
    int(
        resumo_hora[
            "Geracoes_Efetivas"
        ].sum()
    )
    ==
    TOTAL_GERACOES
)


print("\nChecagens:")

print(
    f"Resumo mensal consistente..............: "
    f"{validacao_mensal}"
)

print(
    f"Resumo diário consistente..............: "
    f"{validacao_diaria}"
)

print(
    f"Resumo horário consistente.............: "
    f"{validacao_hora}"
)


# ================================================================
# 49. FINALIZAÇÃO
# ================================================================

print("\n" + "=" * 100)
print("MÓDULO 3 CONCLUÍDO COM SUCESSO")
print("=" * 100)


print(
    f"\nRelatório criado:\n"
    f"{ARQUIVO_RELATORIO}"
)


print(
    "\nPrincipais análises disponíveis:"
)

print(
    """
1. Volumetria mensal
2. Volumetria diária
3. Volumetria horária
4. Dias da semana
5. Períodos do dia
6. Prioridades
7. Categorias
8. Mantenedores
9. Tipos
10. Torres
11. TTL Torre
12. Top dias
13. Top horas
14. Top equipamentos
15. Ranking mensal de equipamentos
16. Categoria × Tipo
17. Mantenedor × Tipo
18. Torre × Tipo
19. Dia da Semana × Hora
20. Mês × Hora
"""
)


# ================================================================
# 50. DOWNLOAD DO RELATÓRIO
# ================================================================

from google.colab import files


print(
    "\nIniciando download do relatório Excel..."
)


files.download(
    ARQUIVO_RELATORIO
)

MÓDULO 3 - ANÁLISE DE VOLUMETRIA DOS ALARMES METASYS

Carregando base do Módulo 2...
Base carregada com sucesso.
Registros: 1,413,882
Colunas: 58

KPIs GERAIS


,Indicador,Quantidade,Percentual_Base
0,Eventos totais,1413882,100.00
1,Eventos operacionais,1320425,93.39
2,Gerações efetivas,637566,45.09
3,Gerações de alarme,240277,16.99
4,Gerações de pré-alarme,296725,20.99
5,Falhas de comunicação,100564,7.11
6,Normalizações de alarme,266494,18.85
7,Normalizações de pré-alarme,263966,18.67
8,Equipamentos únicos,8254,NaN
9,Dias analisados,211,NaN



MÉDIAS GERAIS


,Indicador,Valor
0,Média de eventos totais por dia,"6,700.86"
1,Média de gerações efetivas por dia,"3,021.64"
2,Média de gerações de alarme por dia,"1,138.75"
3,Média de gerações de pré-alarme por dia,"1,406.28"



VOLUMETRIA MENSAL


...


TOP 20 HORAS


,DataHora_Hora,Eventos_Totais,Eventos_Operacionais,Geracoes_Efetivas,Geracoes_Alarme,Geracoes_PreAlarme,Falhas_Comunicacao,Normalizacoes_Alarme,Normalizacoes_PreAlarme,Restabelecimentos_Comunicacao,Eventos_Sistema,Equipamentos_Unicos,Perc_Geracoes_Efetivas,Perc_Eventos_Operacionais
0,2026-01-17 22:00:00,5499,5430,2736,143,10,2583,2577,17,11,13,966,49.75,98.75
1,2026-06-29 06:00:00,4118,3784,1770,222,156,1392,1433,108,73,15,1591,42.98,91.89
2,2026-02-17 17:00:00,3491,3395,1630,220,72,1338,1510,101,35,3,1038,46.69,97.25
3,2026-01-18 01:00:00,2956,2931,1455,91,27,1337,1401,21,4,14,888,49.22,99.15
4,2026-01-22 17:00:00,3005,2869,1417,283,193,941,1083,181,32,15,1455,47.15,95.47
5,2026-01-24 20:00:00,2798,2746,1363,47,11,1305,1302,21,1,10,753,48.71,98.14
6,2026-01-27 15:00:00,3635,3115,1318,261,216,841,970,185,46,13,1871,36.26,85.69
7,2026-01-19 17:00:00,3349,2898,1242,215,162,865,976,139,33,14,1668,37.09,86.53
8,2026-02-16 17:00:00,2962,2676,1223,203,111,909,1019,86,23,16,1373,41.29,90.34
9,2026-06-29 07:00:00,3008,2627,1129,182,119,828,934,114,29,15,1530,37.53,87.33



Gerando relatório Excel do Módulo 3...

VALIDAÇÃO DOS TOTAIS
Eventos totais na base................: 1,413,882
Gerações efetivas na base.............: 637,566
Soma das gerações no resumo mensal....: 637,566
Soma das gerações no resumo diário....: 637,566
Soma das gerações por hora do dia.....: 637,566

Checagens:
Resumo mensal consistente..............: True
Resumo diário consistente..............: True
Resumo horário consistente.............: True

MÓDULO 3 CONCLUÍDO COM SUCESSO

Relatório criado:
Relatorio_Modulo3_Volumetria_Alarmes_Metasys.xlsx

Principais análises disponíveis:

1. Volumetria mensal
2. Volumetria diária
3. Volumetria horária
4. Dias da semana
5. Períodos do dia
6. Prioridades
7. Categorias
8. Mantenedores
9. Tipos
10. Torres
11. TTL Torre
12. Top dias
13. Top horas
14. Top equipamentos
15. Ranking mensal de equipamentos
16. Categoria × Tipo
17. Mantenedor × Tipo
18. Torre × Tipo
19. Dia da Semana × Hora
20. Mês × Hora


Iniciando download do relatório Excel...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>